# CSVW-SAFE DEMO

Install the library with pip

In [1]:
#!pip install csvw-safe

In [1]:
import json
import pandas as pd
from csvw_safe import assert_same_structure, make_dummy_from_metadata, make_metadata_from_data, validate_metadata

In [2]:
import warnings  # pandas deprecation warning to update in library
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("penguin_plus.csv", parse_dates=["timestamp", "timestamp_with_time"])
df.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Torgersen,39.1,18.7,NaN,3750.0,True,7,2025-04-13,2025-04-13 15:04:00,1
1,Adelie,Torgersen,39.5,17.4,NaN,3800.0,False,1,2025-12-15,2025-12-15 18:15:26,2


## Table Level DP

In [4]:
metadata = make_metadata_from_data(df, privacy_unit="penguin_id", with_dependencies=False)
with open("metadata/penguin_metadata_table_level.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [5]:
# (Strict) Pydantic model
validated_metadata = validate_metadata(metadata)

In [6]:
dummy_df = make_dummy_from_metadata(metadata, nb_rows=100, seed=0)
dummy_df.head(5)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,c,j,NaN,21.476563,<NA>,5253,True,185,2025-12-19,2025-02-18 06:47:46,4
1,d,d,42.977825,19.073600,229,5031,False,10,2025-02-12,2025-10-28 06:47:46,0
2,b,a,54.151716,13.444178,211,5575,False,118,2025-07-22,2025-05-14 06:47:46,3
3,d,d,51.422932,14.362348,213,3288,True,261,2025-06-02,2025-05-11 06:47:46,2
4,f,e,54.439223,16.671920,<NA>,5610,True,99,2025-02-18,2025-06-19 06:47:46,4


In [7]:
# Should pass
assert_same_structure(df, dummy_df, check_categories=False)

In [8]:
# Should error as the metadata did not contain categories
assert_same_structure(df, dummy_df, check_categories=True)

AssertionError: Column 'species' dummy values {'a', 'e', 'i', 'h', 'b', 'c', 'f', 'd', 'g', 'j'} are not subset of original {'Adelie', 'Gentoo', 'Chinstrap'}

In [ ]:
# Any combination
dummy_df.groupby(["species", "island"]).groups.keys()

In [9]:
# No ordering (timestamp_with_time after timestamp)
assert (dummy_df["timestamp_with_time"] > dummy_df["timestamp"]).all(), "timestamp_with_time is not always greater than timestamp"

AssertionError: timestamp_with_time is not always greater than timestamp

## Table Level DP with Keys

In [10]:
metadata = make_metadata_from_data(
    df, privacy_unit="penguin_id", default_contributions_level="table_with_keys", with_dependencies=False
)
with open("metadata/penguin_metadata_table_with_keys_level.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [11]:
validated_metadata = validate_metadata(metadata)

In [12]:
dummy_df = make_dummy_from_metadata(metadata, nb_rows=100, seed=0)
dummy_df.head(5)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Torgersen,NaN,21.476563,<NA>,5253,True,185,2025-12-19,2025-02-18 06:47:46,4
1,Chinstrap,Dream,42.977825,19.073600,229,5031,False,10,2025-02-12,2025-10-28 06:47:46,0
2,Adelie,Biscoe,54.151716,13.444178,211,5575,False,118,2025-07-22,2025-05-14 06:47:46,3
3,Adelie,Dream,51.422932,14.362348,213,3288,True,261,2025-06-02,2025-05-11 06:47:46,2
4,Chinstrap,Dream,54.439223,16.671920,<NA>,5610,True,99,2025-02-18,2025-06-19 06:47:46,4


In [13]:
# Should pass
assert_same_structure(df, dummy_df, check_categories=False)
assert_same_structure(df, dummy_df, check_categories=True)

In [14]:
# Any combination
dummy_df.groupby(["species", "island"]).groups.keys()

dict_keys([('Adelie', 'Biscoe'), ('Adelie', 'Dream'), ('Adelie', 'Torgersen'), ('Chinstrap', 'Biscoe'), ('Chinstrap', 'Dream'), ('Chinstrap', 'Torgersen'), ('Gentoo', 'Biscoe'), ('Gentoo', 'Dream'), ('Gentoo', 'Torgersen')])

In [15]:
# No ordering (timestamp_with_time after timestamp)
assert (dummy_df["timestamp_with_time"] > dummy_df["timestamp"]).all(), "timestamp_with_time is not always greater than timestamp"

AssertionError: timestamp_with_time is not always greater than timestamp

## Table Level DP with Keys and Dependencies

In [16]:
metadata = make_metadata_from_data(
    df, default_contributions_level="table_with_keys", privacy_unit="penguin_id", with_dependencies=True
)
with open("metadata/penguin_metadata_table_with_keys_with_dependencies.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [17]:
#metadata

In [18]:
validated_metadata = validate_metadata(metadata)

In [19]:
dummy_df = make_dummy_from_metadata(metadata, nb_rows=100, seed=0)
dummy_df.head(5)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Dream,56.523254,21.476563,<NA>,5253,True,271,2025-11-25,2025-12-30 19:27:11,0
1,Chinstrap,Dream,42.977825,19.073600,176,5031,False,330,2025-05-25,2025-05-30 13:01:31,0
2,Adelie,Torgersen,NaN,13.444178,204,5575,False,219,2025-01-26,2025-02-22 05:51:56,0
3,Gentoo,Biscoe,51.422932,14.362348,199,3288,False,232,2025-05-10,2025-07-05 00:11:08,0
4,Gentoo,Biscoe,NaN,16.671920,<NA>,5610,True,191,2025-06-20,2025-07-13 09:04:16,0


In [20]:
# Only relevant combination
dummy_df.groupby(["species", "island"]).groups.keys()

dict_keys([(np.str_('Adelie'), 'Biscoe'), (np.str_('Adelie'), 'Dream'), (np.str_('Adelie'), 'Torgersen'), (np.str_('Chinstrap'), 'Dream'), (np.str_('Gentoo'), 'Biscoe')])

In [21]:
# No ordering (timestamp_with_time after timestamp)
assert (dummy_df["timestamp_with_time"] >= dummy_df["timestamp"]).all(), "timestamp_with_time is not always greater than timestamp"

## Column Level DP

In [22]:
metadata = make_metadata_from_data(
    df, privacy_unit="penguin_id", default_contributions_level="column", with_dependencies=False
)
with open("metadata/penguin_metadata_column_level.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [23]:
#metadata

In [24]:
validated_metadata = validate_metadata(metadata)

In [25]:
# Looks similar but has DP information
dummy_df = make_dummy_from_metadata(metadata, nb_rows=100, seed=0)
dummy_df.head(3)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Torgersen,NaN,21.476563,<NA>,5253,True,185,2025-12-19,2025-02-18 06:47:46,4
1,Chinstrap,Dream,42.977825,19.073600,229,5031,False,10,2025-02-12,2025-10-28 06:47:46,0
2,Adelie,Biscoe,54.151716,13.444178,211,5575,False,118,2025-07-22,2025-05-14 06:47:46,3


## Partition Level DP

In [26]:
metadata = make_metadata_from_data(
    df, privacy_unit="penguin_id", default_contributions_level="partition", with_dependencies=False
)
with open("metadata/penguin_metadata_partition_level.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [27]:
#metadata

## Partition Level DP with Continuous Partitions

In [28]:
continuous_partitions = {
    "bill_length_mm": [30.0, 40.0, 50.0, 60.0],
    "timestamp": ["2025-01-02 00:00:00", "2025-06-02 00:00:00", "2025-12-30 00:00:00"],
}

In [29]:
metadata = make_metadata_from_data(
    df, 
    privacy_unit="penguin_id", 
    default_contributions_level="partition", 
    continuous_partitions=continuous_partitions, 
    with_dependencies=False
)
with open("metadata/penguin_metadata_partition_level_with_continuous.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [30]:
#metadata

## Per-Column DP Levels

In [31]:
continuous_partitions = {"bill_length_mm": [30.0, 40.0, 50.0, 60.0]}
fine_contrib = {
    "species": "partition",
    "island": "column"
    # "bill_length_mm" is implicitely partition
}

In [32]:
metadata = make_metadata_from_data(
    df, 
    privacy_unit="penguin_id", 
    default_contributions_level="table_with_keys",
    fine_contributions_level=fine_contrib,
    continuous_partitions=continuous_partitions,
    with_dependencies=False
)
with open("metadata/penguin_metadata_fine_contrib_levels.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [33]:
#metadata

## Column Level for Column Groups

In [34]:
column_groups = [
    ["species", "island"],
    ["species", "island", "favourite_number"]
]

In [35]:
metadata = make_metadata_from_data(
    df,
    privacy_unit="penguin_id",
    default_contributions_level="column",
    column_groups=column_groups,
    with_dependencies=False
)
with open("metadata/penguin_metadata_column_level_column_group.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [36]:
#metadata

## Partition Level for Column Groups

In [37]:
column_groups = [["species", "island"]]

In [38]:
metadata = make_metadata_from_data(
    df,
    privacy_unit="penguin_id",
    default_contributions_level="partition",
    column_groups=column_groups,
    with_dependencies=False
)
with open("metadata/penguin_metadata_partition_level_column_group.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [39]:
#metadata

## Partition Level for Column Groups with Continuous Partitions

In [40]:
continuous_partitions = {
    "bill_length_mm": [30.0, 40.0, 50.0, 60.0],
    "timestamp": ["2025-01-02 00:00:00", "2025-06-02 00:00:00", "2025-12-30 00:00:00"],
}
column_groups = [
    ["species", "bill_length_mm"],
]
fine_contrib = {
    "species": "partition",
    "island": "column"
    # "bill_length_mm" is implicitely partition
}

In [41]:
metadata = make_metadata_from_data(
    df,
    privacy_unit="penguin_id",
    default_contributions_level="table_with_keys",
    fine_contributions_level=fine_contrib,
    continuous_partitions=continuous_partitions,
    column_groups=column_groups,
)
with open("metadata/penguin_metadata_fine_levels_column_group_continuous.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [42]:
#metadata

... and any other combination.